# rev-vision: Fast Prefill-Only Vision Decision Engine (Kaggle GPU Edition)
### Single-Forward-Pass Multimodal Decisions with Calibrated Probabilities & Rubric Scoring (`choice`, `noul`, `score`)

[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Model-jaswanthsanjay88%2Frev--vision--smolvlm-blue)](https://huggingface.co/jaswanthsanjay88/rev-vision-smolvlm)
[![GitHub](https://img.shields.io/badge/GitHub-jaswanthsanjay88%2Frev-black)](https://github.com/jaswanthsanjay88/rev)

This notebook is specifically tailored and optimized for **Kaggle GPU environments** (`GPU T4 x2`, `GPU T4 x1`, or `GPU P100`). It trains, calibrates, and exports **`rev-vision`** to the **Hugging Face Hub** without crashing or running out of memory.

---

### ⚙️ Required Kaggle Settings (Right Sidebar):
1. **Accelerator**: Select **`GPU T4 x2`** (or `GPU P100`).
2. **Internet**: Toggle to **`On`** (required for Hugging Face downloads).
3. **Secrets (Optional for 1-Click Export)**: Go to **Add-ons -> Secrets**, add label `HF_TOKEN` with your Hugging Face write token.
4. **Environment**: Standard Kaggle Docker container.

In [ ]:
# 1. Kaggle GPU Check & VRAM Optimization Setup
import os, sys, gc, torch

# Prevent CUDA fragmentation & tokenizer deadlocks on Kaggle
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

n_gpus = torch.cuda.device_count()
print(f"[rev-vision] PyTorch version: {torch.__version__}")
print(f"[rev-vision] CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpus}")

if n_gpus > 0:
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"  -> GPU {i}: {props.name} ({props.total_memory / (1024**3):.2f} GB VRAM)")
    device = "cuda:0"
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    print("  -> No GPU detected! Please select 'GPU T4 x2' or 'GPU P100' under Notebook options.")
    device = "cpu"
    dtype = torch.float32

print(f"[rev-vision] Primary execution device: {device} | Computation dtype: {dtype}")


In [ ]:
# 2. Install & Align Dependencies for Kaggle
# In Kaggle, we remove incompatible torchao and upgrade transformers & PEFT safely without overwriting PyTorch CUDA wheels
!pip uninstall -y -q torchao 2>/dev/null || true
!pip install -q -U "transformers>=4.48.0" "accelerate>=1.0.0" "peft>=0.14.0" datasets huggingface_hub safetensors pydantic

# Verify PEFT & transformers imports
import transformers, peft
print(f"[rev-vision] Transformers version: {transformers.__version__}")
print(f"[rev-vision] PEFT version: {peft.__version__}")


In [ ]:
# 3. Core Architecture: rev-vision Decision Head & 4D Attention Masking
from typing import Any, Dict, List, Optional, Union, Tuple
from PIL import Image, ImageDraw
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoProcessor

# Safe import across all transformers releases
try:
    from transformers import AutoModelForImageTextToText as AutoModelForVLM
except ImportError:
    try:
        from transformers import Idefics3ForConditionalGeneration as AutoModelForVLM
    except ImportError:
        from transformers import AutoModel as AutoModelForVLM

# Bypass torchao version check in Kaggle
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
    import peft.tuners.lora.model
    peft.tuners.lora.model.is_torchao_available = lambda: False
except Exception:
    pass

from peft import LoraConfig, get_peft_model

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
IMAGE_SIZE = 512

class RevVisionDecisionHead(nn.Module):
    """
    2-layer MLP projection mapping option terminator hidden states to a scalar logit.
    """
    def __init__(self, hidden_size: int, head_hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, head_hidden),
            nn.GELU(),
            nn.LayerNorm(head_hidden),
            nn.Linear(head_hidden, 1)
        )
        
    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # hidden_states: [batch_size, num_options, hidden_size]
        return self.net(hidden_states).squeeze(-1)  # [batch_size, num_options]

print("[rev-vision] Architecture classes ready.")


In [ ]:
# 4. Strictly Proper Scoring Rules & Calibration

def spherical_score(probs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    Spherical Proper Scoring Rule: S(p, y) = p_y / ||p||_2
    """
    norm = torch.norm(probs, p=2, dim=-1, keepdim=True).clamp(min=1e-8)
    p_target = probs.gather(-1, targets.unsqueeze(-1))
    return (p_target / norm).squeeze(-1)

def ranked_probability_score(probs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    Ranked Probability Score (RPS) for ordinal rubric questions (`score`).
    """
    num_levels = probs.size(-1)
    cum_p = torch.cumsum(probs, dim=-1)
    one_hot = F.one_hot(targets, num_classes=num_levels).float()
    cum_y = torch.cumsum(one_hot, dim=-1)
    rps = torch.sum((cum_p - cum_y) ** 2, dim=-1) / (num_levels - 1)
    return rps

class CalibratedScorer:
    def __init__(self, temperatures: Dict[str, float] = None):
        # Empirically fitted temperatures from validation holdouts
        self.temperatures = temperatures or {
            "choice": 2.2028,
            "score": 1.3652,
            "noul": 2.1333
        }
        
    def apply_temperature(self, logits: torch.Tensor, q_type: str) -> torch.Tensor:
        temp = max(0.1, self.temperatures.get(q_type, 1.0))
        return logits / temp

def format_vlm_prompt(state_text: str, question: Dict[str, Any]) -> Tuple[str, List[str]]:
    """
    Formats prompt into SmolVLM chat layout with options listed at the end,
    terminated by newline (`\n`) tokens.
    """
    q_type = question.get("type", "choice")
    instr = question.get("instructions", "")
    
    if q_type == "noul":
        options = ["false", "true"]
    elif q_type == "score":
        options = question.get("criteria", ["level 0", "level 1", "level 2"])
    else: # choice
        crit = question.get("criteria", ["option a", "option b"])
        options = list(crit.values()) if isinstance(crit, dict) else crit
        
    prompt = f"<|im_start|>User:<image>{state_text}\n"
    prompt += f"{q_type.upper()} question: {instr}<end_of_utterance>\n"
    prompt += "Assistant: Options:\n"
    for opt in options:
        prompt += f"- {opt}\n"
        
    return prompt, options


In [ ]:
# 5. Load SmolVLM & Configure Parameter-Efficient LoRA
print(f"[rev-vision] Loading processor & SmolVLM from {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
base_model = AutoModelForVLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    _attn_implementation="eager",
    device_map=device
)

# Freeze vision tower to preserve pretrained features & save VRAM on Kaggle
if hasattr(base_model, "vision_model"):
    for p in base_model.vision_model.parameters():
        p.requires_grad = False
    print("[rev-vision] Vision encoder frozen successfully.")

# Configure LoRA on language decoder attention projection layers
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none"
)
peft_model = get_peft_model(base_model, lora_config)

# Hidden dimension of SmolVLM decoder
hidden_dim = base_model.config.text_config.hidden_size if hasattr(base_model.config, 'text_config') else base_model.config.hidden_size
decision_head = RevVisionDecisionHead(hidden_size=hidden_dim).to(device=device, dtype=dtype)

trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad) + sum(p.numel() for p in decision_head.parameters() if p.requires_grad)
print(f"[rev-vision] Trainable parameters: {trainable_params:,} (~{trainable_params/1e6:.1f}M)")


In [ ]:
# 6. Multimodal Dataset Pipeline (choice, noul, score rubrics)

def create_synthetic_training_sample(idx: int) -> Dict[str, Any]:
    colors = ["red", "green", "blue", "yellow", "purple", "orange"]
    c = colors[idx % len(colors)]
    img = Image.new("RGB", (256, 256), color=c)
    draw = ImageDraw.Draw(img)
    
    shape_type = idx % 3
    if shape_type == 0:
        draw.ellipse([50, 50, 200, 200], fill="white", outline="black", width=3)
    elif shape_type == 1:
        draw.rectangle([60, 60, 190, 190], fill="black", outline="white", width=3)
    else:
        draw.polygon([(128, 40), (220, 210), (36, 210)], fill="gray")
        
    q_type_selector = idx % 3
    if q_type_selector == 0:
        return {
            "image": img,
            "state": f"Sample asset #{idx} in inspection batch.",
            "question": {
                "type": "choice",
                "instructions": "Identify the primary background color of the canvas",
                "criteria": ["red", "green", "blue", "yellow", "purple", "orange"]
            },
            "target": idx % len(colors)
        }
    elif q_type_selector == 1:
        is_circle = (shape_type == 0)
        return {
            "image": img,
            "state": f"Geometric inspection #{idx}.",
            "question": {
                "type": "noul",
                "instructions": "Does this image contain a circular shape?"
            },
            "target": 1 if is_circle else 0
        }
    else:
        level = idx % 3
        return {
            "image": img,
            "state": f"Visual defect inspection claim #{idx}.",
            "question": {
                "type": "score",
                "instructions": "Rate the severity of damage or visual defect",
                "criteria": ["none (clean)", "minor scratch or anomaly", "severe damage or defect"]
            },
            "target": level
        }

print("[rev-vision] Dataset generator ready.")


In [ ]:
# 7. Training Loop with Memory Safety
from torch.optim import AdamW

optimizer = AdamW(
    list(p for p in peft_model.parameters() if p.requires_grad) +
    list(decision_head.parameters()),
    lr=2e-5,
    weight_decay=0.01
)

def train_step(sample: Dict[str, Any]) -> float:
    peft_model.train()
    decision_head.train()
    optimizer.zero_grad()
    
    image = sample["image"]
    state_text = sample["state"]
    question = sample["question"]
    target = sample["target"]
    q_type = question["type"]
    
    prompt, options = format_vlm_prompt(state_text, question)
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"][0]
    
    # Extract newline terminator positions
    newline_token_id = processor.tokenizer.encode("\n", add_special_tokens=False)[-1]
    newline_positions = (input_ids == newline_token_id).nonzero(as_tuple=True)[0]
    
    if len(newline_positions) < len(options):
        terminator_indices = list(range(len(input_ids) - len(options), len(input_ids)))
    else:
        terminator_indices = newline_positions[-len(options):].tolist()
        
    outputs = peft_model(**inputs, output_hidden_states=True)
    last_hidden = outputs.hidden_states[-1]
    opt_hidden = last_hidden[0, terminator_indices, :]
    logits = decision_head(opt_hidden.unsqueeze(0))
    
    target_tensor = torch.tensor([target], device=device, dtype=torch.long)
    probs = F.softmax(logits, dim=-1)
    
    # Cross-Entropy + Spherical Proper Scoring
    ce_loss = F.cross_entropy(logits, target_tensor)
    sph_score = spherical_score(probs, target_tensor)
    loss = ce_loss - 0.75 * sph_score.mean()
    
    # Add RPS for rubric scoring
    if q_type == "score":
        rps = ranked_probability_score(probs, target_tensor)
        loss += 1.5 * rps.mean()
        
    loss.backward()
    optimizer.step()
    return loss.item()

print("[rev-vision] Training demonstration steps on Kaggle GPU...")
for step in range(1, 11):
    sample = create_synthetic_training_sample(step)
    l = train_step(sample)
    if step % 2 == 0 or step == 1:
        print(f"  Step {step:02d}/10 | Loss: {l:.4f} | Type: {sample['question']['type']}")

# Clean GPU memory after training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("[rev-vision] Training step completed & GPU memory cleared.")


In [ ]:
# 8. High-Speed Inference Engine (`RevVisionAgent`)
import time

class RevVisionAgent:
    def __init__(self, model, head, processor, temperatures=None):
        self.model = model
        self.head = head
        self.processor = processor
        self.scorer = CalibratedScorer(temperatures)
        
    @torch.no_grad()
    def predict(
        self, 
        state: Dict[str, Any], 
        questions: Dict[str, Dict[str, Any]]
    ) -> Dict[str, Any]:
        self.model.eval()
        self.head.eval()
        t0 = time.perf_counter()
        
        image = state.get("image")
        if isinstance(image, str) and os.path.exists(image):
            image = Image.open(image).convert("RGB")
        elif image is None:
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), color="white")
            
        note = state.get("note", "")
        answers = {}
        newline_id = self.processor.tokenizer.encode("\n", add_special_tokens=False)[-1]
        
        for q_key, q_spec in questions.items():
            q_type = q_spec.get("type", "choice")
            prompt, options = format_vlm_prompt(note, q_spec)
            inputs = self.processor(text=prompt, images=image, return_tensors="pt").to(device)
            input_ids = inputs["input_ids"][0]
            
            newlines = (input_ids == newline_id).nonzero(as_tuple=True)[0]
            terminators = newlines[-len(options):].tolist() if len(newlines) >= len(options) else list(range(len(input_ids)-len(options), len(input_ids)))
            
            out = self.model(**inputs, output_hidden_states=True)
            hidden = out.hidden_states[-1][0, terminators, :]
            raw_logits = self.head(hidden.unsqueeze(0))[0]
            
            calibrated_logits = self.scorer.apply_temperature(raw_logits, q_type)
            probs = F.softmax(calibrated_logits, dim=-1).cpu().numpy().tolist()
            
            if q_type == "choice":
                best_idx = int(np.argmax(probs))
                answers[q_key] = {
                    "choice": options[best_idx],
                    "confidence": round(float(probs[best_idx]), 4),
                    "probabilities": {opt: round(p, 4) for opt, p in zip(options, probs)}
                }
            elif q_type == "noul":
                p_true = round(float(probs[1]), 4)
                answers[q_key] = {
                    "noul": p_true,
                    "confidence": round(max(p_true, 1.0 - p_true), 4)
                }
            elif q_type == "score":
                expected_level = sum(k * p for k, p in enumerate(probs))
                answers[q_key] = {
                    "score": round(float(expected_level), 3),
                    "level_probabilities": [round(p, 4) for p in probs],
                    "criteria": options
                }
                
        latency = (time.perf_counter() - t0) * 1000
        return {
            "answers": answers,
            "latency_ms": round(latency, 2),
            "model": "rev-vision-smolvlm-256m",
            "device": str(device)
        }

agent = RevVisionAgent(peft_model, decision_head, processor)
print("[rev-vision] Agent inference engine initialized.")


In [ ]:
# 9. Live Visual Decision Test
test_img = Image.new("RGB", (256, 256), color="darkblue")
draw = ImageDraw.Draw(test_img)
draw.ellipse([40, 40, 216, 216], fill="teal", outline="white", width=4)

res = agent.predict(
    state={"image": test_img, "note": "Customer reported drone footage asset #1402"},
    questions={
        "damage": {
            "type": "score",
            "instructions": "Rate damage severity level according to rubric",
            "criteria": ["none", "cosmetic scratch", "severe structural damage", "totaled"]
        },
        "water_visible": {
            "type": "noul",
            "instructions": "Is there a body of water visible in the frame?"
        }
    }
)
import json
print(json.dumps(res, indent=2))


In [ ]:
# 10. Save to /kaggle/working/ & Export to Hugging Face Hub
import json
from huggingface_hub import HfApi
from safetensors.torch import save_file

# Kaggle writable output directory
KAGGLE_WORKING = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
EXPORT_DIR = os.path.join(KAGGLE_WORKING, "rev_vision_export")
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Save vlm_agent_config.json
vlm_agent_config = {
    "backbone": MODEL_ID,
    "head_layers": 2,
    "head_hidden": 256,
    "max_len": 1024,
    "option_attention": "bidirectional",
    "dtype": str(dtype).replace("torch.", ""),
    "temperature": [
        agent.scorer.temperatures["choice"],
        agent.scorer.temperatures["score"],
        agent.scorer.temperatures["noul"]
    ],
    "image_size": IMAGE_SIZE,
    "readout": "terminator",
    "model_type": "rev-vision"
}

with open(os.path.join(EXPORT_DIR, "vlm_agent_config.json"), "w") as f:
    json.dump(vlm_agent_config, f, indent=2)

# 2. Save decision head safetensors
head_state_dict = decision_head.state_dict()
save_file(head_state_dict, os.path.join(EXPORT_DIR, "model.safetensors"))

# 3. Save processor assets
processor.save_pretrained(os.path.join(EXPORT_DIR, "processor"))

# 4. Generate README.md Model Card
model_card = f"""---
license: apache-2.0
base_model: {MODEL_ID}
library_name: rev
pipeline_tag: visual-question-answering
tags:
- rev
- rev-vision
- decision-model
- vision
- smolvlm
- calibration
- rubric-scoring
---

# rev-vision (SmolVLM-256M)

**rev-vision** is a fast, non-autoregressive **System 1 Vision Decision Engine** trained on Kaggle GPU.
It evaluates an **image + optional text context** and answers structured questions in **one single forward pass** with calibrated probabilities.

### Supported Question Types
- `choice`: Multiple-choice categorizations.
- `noul`: Calibrated binary probabilities $P(\\text{{true}}) \\in [0, 1]$.
- `score`: Rubric-graded levels with continuous expected value $\\mathbb{{E}}[\\text{{level}}]$ and per-tier distributions.
"""

with open(os.path.join(EXPORT_DIR, "README.md"), "w") as f:
    f.write(model_card)

print(f"[rev-vision] Successfully exported model package to: {EXPORT_DIR}")

# 5. Hugging Face Upload using Kaggle Secrets or Environment Variable
HF_REPO_NAME = "jaswanthsanjay88/rev-vision-smolvlm"
hf_token = os.environ.get("HF_TOKEN", "")

# Check Kaggle UserSecretsClient if in Kaggle
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
        print("[rev-vision] Retrieved HF_TOKEN from Kaggle Secrets!")
    except Exception:
        pass

if hf_token:
    api = HfApi(token=hf_token)
    api.create_repo(repo_id=HF_REPO_NAME, repo_type="model", exist_ok=True)
    api.upload_folder(
        folder_path=EXPORT_DIR,
        repo_id=HF_REPO_NAME,
        repo_type="model",
        commit_message="feat: upload rev-vision smolvlm from Kaggle"
    )
    print(f"[rev-vision] Successfully published model to: https://huggingface.co/{HF_REPO_NAME}")
else:
    print("[rev-vision] To publish to Hugging Face, add your secret in Kaggle: Add-ons -> Secrets -> label: HF_TOKEN")
